# Task #57 — Huấn luyện lại 3 model với đặc trưng khoảng cách, so sánh trước/sau (Story #10)

`orders_features_train.csv`/`orders_features_test.csv` giờ có thêm `seller_customer_distance_km` (notebook #23, 76 đặc trưng thay vì 75). Notebook này huấn luyện lại 3 model **giữ nguyên cấu hình đã dùng ở Task #49** (`class_weight`/`scale_pos_weight` từ Task #47 — không đổi vì thêm đặc trưng không ảnh hưởng phân phối nhãn, `random_state=42`), đánh giá trên test giống Task #50 + notebook #22 (F1/Precision/Recall ở ngưỡng 0.5, F1 tối đa qua tinh ngưỡng, PR-AUC), rồi so sánh trực tiếp với kết quả "trước" đã lưu ở `models/evaluation_results.csv` và `models/threshold_diagnostics.csv`.

**Không ghi đè model/kết quả gốc** — lưu vào file riêng hậu tố `_with_distance` để giữ nguyên baseline Task #49/#50 làm mốc so sánh.

## 1. Huấn luyện 3 model trên tập train (đã có đặc trưng khoảng cách)

In [1]:
import time
import json
from pathlib import Path

import joblib
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

df = pd.read_csv("../data/processed/orders_features_train.csv", low_memory=False)

df["is_delayed"] = df["is_delayed"].astype(bool)

bool_cols = ["payment_has_boleto", "payment_has_credit_card", "payment_has_debit_card",
             "payment_has_not_defined", "payment_has_voucher", "items_multi_seller"]
for col in bool_cols:
    df[col] = df[col].astype("boolean")

X_train = df.drop(columns=["order_id", "is_delayed"])
y_train = df["is_delayed"].astype(int)

print("X_train:", X_train.shape, " y_train:", y_train.shape)
assert "seller_customer_distance_km" in X_train.columns

X_train: (77156, 76)  y_train: (77156,)


In [2]:
class_weight_dict = {False: 0.5441568516820651, True: 6.161635521482191}
scale_pos_weight = 11.3233
class_weight_sklearn = {0: class_weight_dict[False], 1: class_weight_dict[True]}

training_times = {}

start = time.perf_counter()
log_reg = LogisticRegression(class_weight=class_weight_sklearn, max_iter=1000, random_state=42)
log_reg.fit(X_train, y_train)
training_times["logistic_regression"] = time.perf_counter() - start
print(f"Logistic Regression: {training_times['logistic_regression']:.2f}s")

start = time.perf_counter()
rand_forest = RandomForestClassifier(class_weight=class_weight_sklearn, random_state=42, n_jobs=-1)
rand_forest.fit(X_train, y_train)
training_times["random_forest"] = time.perf_counter() - start
print(f"Random Forest: {training_times['random_forest']:.2f}s")

start = time.perf_counter()
xgb = XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=42, n_jobs=-1, eval_metric="logloss")
xgb.fit(X_train, y_train)
training_times["xgboost"] = time.perf_counter() - start
print(f"XGBoost: {training_times['xgboost']:.2f}s")

D:\Project Management\delivery-performance-intelligence\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic Regression: 20.37s


Random Forest: 2.14s


XGBoost: 2.44s


In [3]:
models_dir = Path("../models")

joblib.dump(log_reg, models_dir / "logistic_regression_with_distance.pkl")
joblib.dump(rand_forest, models_dir / "random_forest_with_distance.pkl")
joblib.dump(xgb, models_dir / "xgboost_with_distance.pkl")

with open(models_dir / "training_times_with_distance.json", "w", encoding="utf-8") as f:
    json.dump(training_times, f, indent=2)

print("Da luu 3 model va training_times_with_distance.json vao models/")

Da luu 3 model va training_times_with_distance.json vao models/


## 2. Đánh giá trên tập test: F1/Precision/Recall (ngưỡng 0.5), F1 tối đa qua tinh ngưỡng, PR-AUC

In [4]:
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    precision_recall_curve, average_precision_score,
)

test_df = pd.read_csv("../data/processed/orders_features_test.csv", low_memory=False)
test_df["is_delayed"] = test_df["is_delayed"].astype(bool)
for col in bool_cols:
    test_df[col] = test_df[col].astype("boolean")

X_test = test_df.drop(columns=["order_id", "is_delayed"])
y_test = test_df["is_delayed"].astype(int)

models = {"logistic_regression": log_reg, "random_forest": rand_forest, "xgboost": xgb}

eval_rows = []
threshold_rows = []
for name, model in models.items():
    y_score = model.predict_proba(X_test)[:, 1]
    y_pred = (y_score >= 0.5).astype(int)

    eval_rows.append({
        "model": name,
        "precision": precision_score(y_test, y_pred, pos_label=1),
        "recall": recall_score(y_test, y_pred, pos_label=1),
        "f1": f1_score(y_test, y_pred, pos_label=1),
        "training_time_s": training_times[name],
    })

    precision, recall, thresholds = precision_recall_curve(y_test, y_score)
    f1_per_threshold = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    best_idx = f1_per_threshold.argmax()
    threshold_rows.append({
        "model": name,
        "f1_at_default_0.5": f1_score(y_test, y_pred),
        "max_f1_via_threshold": f1_per_threshold[best_idx],
        "best_threshold": thresholds[best_idx],
        "precision_at_best": precision[best_idx],
        "recall_at_best": recall[best_idx],
        "average_precision_pr_auc": average_precision_score(y_test, y_score),
    })

eval_with_distance_df = pd.DataFrame(eval_rows).sort_values("f1", ascending=False).reset_index(drop=True)
threshold_with_distance_df = pd.DataFrame(threshold_rows).sort_values("max_f1_via_threshold", ascending=False).reset_index(drop=True)

eval_with_distance_df

,model,precision,recall,f1,training_time_s
0,xgboost,0.215668,0.598083,0.317019,2.441557
1,random_forest,0.418470,0.185304,0.256864,2.138089
2,logistic_regression,0.121870,0.677955,0.206601,20.366829


## 3. Feature importance — `seller_customer_distance_km` xếp hạng bao nhiêu?

In [5]:
importance_frames = []
for name, model in [("random_forest", rand_forest), ("xgboost", xgb)]:
    imp_df = pd.DataFrame({
        "model": name,
        "feature": X_test.columns,
        "importance": model.feature_importances_,
    }).sort_values("importance", ascending=False)
    imp_df["rank"] = range(1, len(imp_df) + 1)
    importance_frames.append(imp_df)

importance_with_distance_df = pd.concat(importance_frames, ignore_index=True)

for name in ["random_forest", "xgboost"]:
    row = importance_with_distance_df[
        (importance_with_distance_df["model"] == name)
        & (importance_with_distance_df["feature"] == "seller_customer_distance_km")
    ]
    print(f"{name}: seller_customer_distance_km rank {row['rank'].values[0]} / {len(X_test.columns)}, "
          f"importance {row['importance'].values[0]:.4f}")

importance_with_distance_df[importance_with_distance_df["model"] == "xgboost"].head(10)[["rank", "feature", "importance"]]

random_forest: seller_customer_distance_km rank 3 / 76, importance 0.1013
xgboost: seller_customer_distance_km rank 11 / 76, importance 0.0176


,rank,feature,importance
76,1,customer_state_SP,0.112841
77,2,payment_has_credit_card,0.061779
78,3,items_num_sellers,0.050938
79,4,customer_state_RJ,0.043152
80,5,customer_state_PR,0.041888
81,6,order_purchase_month,0.039881
82,7,customer_state_MG,0.039763
83,8,estimated_delivery_days,0.024781
84,9,primary_seller_state_MA,0.020408
85,10,primary_seller_state_SP,0.019895


## 4. Lưu kết quả mới, so sánh trực tiếp với baseline (Task #50 / notebook #22)

In [6]:
eval_with_distance_df.to_csv(models_dir / "evaluation_results_with_distance.csv", index=False)
threshold_with_distance_df.to_csv(models_dir / "threshold_diagnostics_with_distance.csv", index=False)
importance_with_distance_df.to_csv(models_dir / "feature_importance_with_distance.csv", index=False)
print("Da luu evaluation_results_with_distance.csv, threshold_diagnostics_with_distance.csv, feature_importance_with_distance.csv")

baseline_eval = pd.read_csv(models_dir / "evaluation_results.csv")
baseline_threshold = pd.read_csv(models_dir / "threshold_diagnostics.csv")

compare = baseline_eval[["model", "f1"]].rename(columns={"f1": "f1_before"}).merge(
    eval_with_distance_df[["model", "f1"]].rename(columns={"f1": "f1_after"}), on="model"
).merge(
    baseline_threshold[["model", "max_f1_via_threshold", "average_precision_pr_auc"]].rename(
        columns={"max_f1_via_threshold": "max_f1_before", "average_precision_pr_auc": "pr_auc_before"}
    ), on="model"
).merge(
    threshold_with_distance_df[["model", "max_f1_via_threshold", "average_precision_pr_auc"]].rename(
        columns={"max_f1_via_threshold": "max_f1_after", "average_precision_pr_auc": "pr_auc_after"}
    ), on="model"
)
compare["f1_delta"] = compare["f1_after"] - compare["f1_before"]
compare["pr_auc_delta"] = compare["pr_auc_after"] - compare["pr_auc_before"]
compare = compare.sort_values("f1_after", ascending=False).reset_index(drop=True)
compare

Da luu evaluation_results_with_distance.csv, threshold_diagnostics_with_distance.csv, feature_importance_with_distance.csv


,model,f1_before,f1_after,max_f1_before,pr_auc_before,max_f1_after,pr_auc_after,f1_delta,pr_auc_delta
0,xgboost,0.311967,0.317019,0.353143,0.280744,0.350995,0.285490,0.005053,0.004746
1,random_forest,0.260908,0.256864,0.339378,0.261143,0.335065,0.271570,-0.004043,0.010427
2,logistic_regression,0.200802,0.206601,0.223906,0.148707,0.234006,0.156366,0.005799,0.007660
